# BINN-as-GNN on TCGA (Phases 7–9): Training + Comparisons + Visualizations

This notebook continues after you have already built and cached:
- TCGA expression (`expr_reactome_tcga_tumor_normal.parquet`) and labels (`y_tcga_tumor_normal.csv`)
- Layered/padded Reactome graph artifacts (`graph_layered_binn/`), including:
  - `edge_index_by_layer.pt`, `edge_id_by_layer.pt`, `dst_nodes_by_layer.pt`
  - `root_pathway_idx.pt`
  - `gene_layered_ids_in_expr_order.pt`

## What we do here
### Phase 7 (Replica)
Train **BINN-exact** as a special case of a GNN:
- scalar node states (d=1)
- **unique learnable weight per edge**
- sequential layer-by-layer propagation (feedforward schedule)
- read out from **29 root pathways** → classify tumor vs normal

### Phase 8 (Relaxations)
Train a few GNN variants that relax BINN assumptions:
- **GCN-shared**: shared weights (per-layer) instead of unique per-edge weights
- **Gated attention (GAT-like)**: per-edge dynamic gate (sample-dependent), still sequential
- **Vector embeddings (d>1)**: pathway nodes carry vectors instead of scalars (shared weights)

### Phase 9 (Baselines)
Train non-biological baselines for comparison:
- **Dense MLP** on gene expression only (no graph)

## Outputs
- Saved checkpoints for each model
- Comparison plots: loss curves, ROC/PR curves, bar charts of metrics, confusion matrices

> Default settings are "quick-run" (few epochs). Increase `EPOCHS` and tune hyperparameters for serious runs.

In [ ]:
# ==== 0) Setup: paths and run configuration ====
from pathlib import Path
import json, os, time
import numpy as np
import pandas as pd
import torch

# If you're in Colab with Drive mounted, this should match your previous notebooks.
BASE_DIR = Path("/content/drive/MyDrive/binn_gnn_tcga")

OUT_DIR = BASE_DIR / "outputs"
GRAPH_LAYER_DIR = OUT_DIR / "graph_layered_binn"

# Cached dataset
EXPR_PARQUET = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
Y_CSV = OUT_DIR / "y_tcga_tumor_normal.csv"

# Artifacts from Phase 3–4
EDGE_INDEX_BY_LAYER_PT = GRAPH_LAYER_DIR / "edge_index_by_layer.pt"
EDGE_ID_BY_LAYER_PT = GRAPH_LAYER_DIR / "edge_id_by_layer.pt"
DST_NODES_BY_LAYER_PT = GRAPH_LAYER_DIR / "dst_nodes_by_layer.pt"
ROOT_PATHWAY_IDX_PT = GRAPH_LAYER_DIR / "root_pathway_idx.pt"
GENE_LAYERED_IDS_PT = GRAPH_LAYER_DIR / "gene_layered_ids_in_expr_order.pt"

# Cache to speed up future runs
CACHE_DIR = OUT_DIR / "cache_tensors"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
X_GENE_PT = CACHE_DIR / "X_gene_samples_by_genes.pt"   # [S, G] float32
Y_PT = CACHE_DIR / "y.pt"                              # [S] int64
SAMPLES_JSON = CACHE_DIR / "sample_order.json"         # list of sample ids in X

# Training settings (quick-run defaults)
SEED = 42
EPOCHS = 3
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2

# Models to run (edit this list)
MODELS_TO_RUN = [
    "binn_exact_d1",       # Phase 7 replica
    "gcn_shared_d1",       # Phase 8 relaxation
    "gat_gate_d1",         # Phase 8 relaxation
    "vector_shared_d16",   # Phase 8 relaxation (d>1)
    "dense_mlp_baseline",  # Phase 9 baseline
]

# Vector size for the d>1 model
D_VECTOR = 16

def set_seed(seed:int=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Sanity checks: required files exist
required = [EXPR_PARQUET, Y_CSV, EDGE_INDEX_BY_LAYER_PT, EDGE_ID_BY_LAYER_PT, DST_NODES_BY_LAYER_PT, ROOT_PATHWAY_IDX_PT, GENE_LAYERED_IDS_PT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("All required inputs found.")

## Phase 7–9, Section 1: Load cached artifacts and build fast tensors

We load the layered graph schedule and build:
- `X_gene` tensor of shape **[num_samples, num_genes]**
- `y` tensor of shape **[num_samples]**

We also compute a stratified train/val split and per-gene normalization stats.

In [ ]:
# ==== 1) Load layered graph schedule artifacts ====
edge_index_by_layer = torch.load(EDGE_INDEX_BY_LAYER_PT)  # list of [2, E_l]
edge_id_by_layer = torch.load(EDGE_ID_BY_LAYER_PT)        # list of [E_l]
dst_nodes_by_layer = torch.load(DST_NODES_BY_LAYER_PT)    # list of [num_dst_l]
root_idx = torch.load(ROOT_PATHWAY_IDX_PT).long()         # [R]
gene_layered_ids = torch.load(GENE_LAYERED_IDS_PT).long() # [G] in expr row order

L = len(edge_index_by_layer)
R = int(root_idx.numel())
G = int(gene_layered_ids.numel())

# Infer N_layered and E_layered
N_layered = int(max([int(ei.max()) for ei in edge_index_by_layer]) + 1)
E_layered = int(max([int(eid.max()) for eid in edge_id_by_layer]) + 1)

print(f"L={L}  R={R}  G={G}  N_layered={N_layered}  E_layered={E_layered}")
print("Layer edge counts:", [int(ei.size(1)) for ei in edge_index_by_layer])

In [ ]:
# ==== 2) Load (or build) X_gene and y tensors ====
# X_gene will be [S, G] float32 where columns follow expr.index order used in Phase 4 mapping.
# This block caches to .pt to avoid parquet overhead next time.

if X_GENE_PT.exists() and Y_PT.exists() and SAMPLES_JSON.exists():
    X_gene = torch.load(X_GENE_PT, map_location="cpu")
    y = torch.load(Y_PT, map_location="cpu")
    sample_order = json.loads(SAMPLES_JSON.read_text())
    print("Loaded cached tensors:")
    print(" - X_gene:", tuple(X_gene.shape), X_gene.dtype)
    print(" - y:", tuple(y.shape), y.dtype)
else:
    print("Building tensors from parquet/csv (first run may take a bit)...")
    expr = pd.read_parquet(EXPR_PARQUET)  # genes x samples
    y_series = pd.read_csv(Y_CSV, index_col=0).iloc[:,0]  # sample -> y

    # Align columns to y and keep same sample order
    common_samples = [s for s in expr.columns if s in y_series.index]
    expr = expr.loc[:, common_samples]
    y_series = y_series.loc[common_samples].astype(int)

    # Convert to [S, G]
    X_gene = torch.tensor(expr.to_numpy(dtype=np.float32).T, dtype=torch.float32)  # [S, G]
    y = torch.tensor(y_series.to_numpy(dtype=np.int64), dtype=torch.long)          # [S]
    sample_order = common_samples

    torch.save(X_gene, X_GENE_PT)
    torch.save(y, Y_PT)
    SAMPLES_JSON.write_text(json.dumps(sample_order))
    print("Saved cached tensors to:", CACHE_DIR)

print("Final tensors:")
print(" - X_gene:", tuple(X_gene.shape), X_gene.dtype)
print(" - y:", tuple(y.shape), y.dtype)

# Class balance
counts = torch.bincount(y)
print("Class counts:", {int(i): int(c) for i,c in enumerate(counts)})

In [ ]:
# ==== 3) Train/val split + standardization (train stats only) ====
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y.numpy())

X_train = X_gene[train_idx]
y_train = y[train_idx]
X_val = X_gene[val_idx]
y_val = y[val_idx]

# Per-gene standardization using TRAIN ONLY (recommended for tanh stability)
train_mean = X_train.mean(dim=0, keepdim=True)
train_std = X_train.std(dim=0, keepdim=True).clamp_min(1e-6)

def standardize(X):
    return (X - train_mean) / train_std

X_train = standardize(X_train)
X_val = standardize(X_val)

print("Train:", tuple(X_train.shape), "Val:", tuple(X_val.shape))
print("Train class counts:", {int(i): int(c) for i,c in enumerate(torch.bincount(y_train))})
print("Val class counts:", {int(i): int(c) for i,c in enumerate(torch.bincount(y_val))})

# Class weights (inverse frequency) for imbalanced TCGA
train_counts = torch.bincount(y_train)
class_weights = (1.0 / train_counts.float()).to(device)
class_weights = class_weights / class_weights.sum() * 2.0  # rescale to sum=2
print("Class weights:", class_weights.detach().cpu().numpy())

## Phase 7–9, Section 2: Define models

All graph-based models share the same data injection:
- build `h0` with gene expression on gene nodes
- pathway nodes start at 0
- sequentially propagate layer-by-layer
- read out the **29 root pathway nodes** (flattened)

Models:
- **BINN-exact (d=1, unique edge weights)**
- **GCN-shared (d=1, shared per-layer scalar weights)**
- **GAT-gate (d=1, per-edge dynamic gate)**
- **Vector shared (d=16)**
- **Dense MLP baseline (no graph)**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Precompute per-layer structures for fast aggregation into dst-unique lists.
# For each layer we create:
# - src_idx [E]
# - dst_unique [U]
# - dst_pos [E] where dst_pos maps each edge's dst into [0..U-1]
# - edge_id [E] (for BINN-exact)
layer_structs = []
for l in range(L):
    ei = edge_index_by_layer[l].long()
    src = ei[0]
    dst = ei[1]
    eid = edge_id_by_layer[l].long()

    dst_unique, inv = torch.unique(dst, sorted=True, return_inverse=True)
    # inv: [E] indices into dst_unique
    layer_structs.append({
        "src": src,
        "dst_unique": dst_unique,
        "dst_pos": inv,
        "eid": eid,
    })

# Padding edges: in Phase 2/3 you marked padding edges in edge_table; in this training notebook we assume
# padding weights are learnable like other edges. If you want strict identity padding, you can add a mask later.

class LayeredMPNNBase(nn.Module):
    def __init__(self, N_layered:int, L:int, root_idx:torch.Tensor, dropout:float=0.0):
        super().__init__()
        self.N = N_layered
        self.L = L
        self.root_idx = root_idx.clone().long()
        self.dropout = nn.Dropout(dropout)

    def readout_roots_flat(self, h):
        # h: [B, N] or [B, N, D]
        roots = h.index_select(1, self.root_idx)  # [B, R] or [B, R, D]
        return roots.reshape(roots.size(0), -1)    # flatten

class BINNExactD1(LayeredMPNNBase):
    def __init__(self, N_layered, E_layered, L, root_idx, dropout=0.0):
        super().__init__(N_layered, L, root_idx, dropout)
        self.edge_weight = nn.Parameter(torch.zeros(E_layered))
        nn.init.normal_(self.edge_weight, mean=0.0, std=0.01)
        self.node_bias = nn.Parameter(torch.zeros(N_layered))

        self.head = nn.Linear(root_idx.numel(), 2)

    def forward(self, X_gene_batch):
        # X_gene_batch: [B, G] already standardized
        B = X_gene_batch.size(0)
        h = torch.zeros((B, self.N), device=X_gene_batch.device, dtype=X_gene_batch.dtype)
        h[:, gene_layered_ids.to(X_gene_batch.device)] = X_gene_batch

        for l in range(self.L):
            st = layer_structs[l]
            src = st["src"].to(X_gene_batch.device)
            dst_unique = st["dst_unique"].to(X_gene_batch.device)
            dst_pos = st["dst_pos"].to(X_gene_batch.device)
            eid = st["eid"].to(X_gene_batch.device)

            w = self.edge_weight.index_select(0, eid)  # [E]
            msg = h.index_select(1, src) * w.view(1, -1)  # [B, E]
            agg = torch.zeros((B, dst_unique.numel()), device=h.device, dtype=h.dtype)
            agg.index_add_(1, dst_pos, msg)  # sum into dst_unique bins

            h_dst = torch.tanh(agg + self.node_bias.index_select(0, dst_unique).view(1, -1))
            h_dst = self.dropout(h_dst)
            h[:, dst_unique] = h_dst

        feat = self.readout_roots_flat(h)  # [B, 29]
        return self.head(feat)

class GCNSharedD1(LayeredMPNNBase):
    def __init__(self, N_layered, L, root_idx, dropout=0.0):
        super().__init__(N_layered, L, root_idx, dropout)
        # One scalar weight per layer (shared across edges)
        self.layer_weight = nn.Parameter(torch.ones(L) * 0.01)
        self.node_bias = nn.Parameter(torch.zeros(N_layered))
        self.head = nn.Linear(root_idx.numel(), 2)

    def forward(self, X_gene_batch):
        B = X_gene_batch.size(0)
        h = torch.zeros((B, self.N), device=X_gene_batch.device, dtype=X_gene_batch.dtype)
        h[:, gene_layered_ids.to(X_gene_batch.device)] = X_gene_batch

        for l in range(self.L):
            st = layer_structs[l]
            src = st["src"].to(h.device)
            dst_unique = st["dst_unique"].to(h.device)
            dst_pos = st["dst_pos"].to(h.device)

            w = self.layer_weight[l]
            msg = h.index_select(1, src) * w  # [B, E]
            agg = torch.zeros((B, dst_unique.numel()), device=h.device, dtype=h.dtype)
            agg.index_add_(1, dst_pos, msg)

            h_dst = torch.tanh(agg + self.node_bias.index_select(0, dst_unique).view(1, -1))
            h_dst = self.dropout(h_dst)
            h[:, dst_unique] = h_dst

        feat = self.readout_roots_flat(h)
        return self.head(feat)

class GATGateD1(LayeredMPNNBase):
    def __init__(self, N_layered, L, root_idx, dropout=0.0):
        super().__init__(N_layered, L, root_idx, dropout)
        # Shared weight per layer + a simple gate per edge computed from (h_src, h_dst_prev)
        self.layer_weight = nn.Parameter(torch.ones(L) * 0.01)
        self.a_src = nn.Parameter(torch.zeros(L))  # gate coefficient for h_src
        self.a_dst = nn.Parameter(torch.zeros(L))  # gate coefficient for h_dst
        self.node_bias = nn.Parameter(torch.zeros(N_layered))
        self.head = nn.Linear(root_idx.numel(), 2)

    def forward(self, X_gene_batch):
        B = X_gene_batch.size(0)
        h = torch.zeros((B, self.N), device=X_gene_batch.device, dtype=X_gene_batch.dtype)
        h[:, gene_layered_ids.to(h.device)] = X_gene_batch

        for l in range(self.L):
            st = layer_structs[l]
            src = st["src"].to(h.device)
            dst = edge_index_by_layer[l][1].to(h.device)  # original dst list (not unique)
            dst_unique = st["dst_unique"].to(h.device)
            dst_pos = st["dst_pos"].to(h.device)

            w = self.layer_weight[l]
            h_src = h.index_select(1, src)    # [B, E]
            h_dst_prev = h.index_select(1, dst)  # [B, E] (mostly zeros for new layer, but ok)

            gate = torch.sigmoid(self.a_src[l] * h_src + self.a_dst[l] * h_dst_prev)  # [B, E]
            msg = h_src * (w * gate)  # [B, E]
            agg = torch.zeros((B, dst_unique.numel()), device=h.device, dtype=h.dtype)
            agg.index_add_(1, dst_pos, msg)

            h_dst = torch.tanh(agg + self.node_bias.index_select(0, dst_unique).view(1, -1))
            h_dst = self.dropout(h_dst)
            h[:, dst_unique] = h_dst

        feat = self.readout_roots_flat(h)
        return self.head(feat)

class VectorSharedD(LayeredMPNNBase):
    def __init__(self, N_layered, L, root_idx, d:int=16, dropout=0.0):
        super().__init__(N_layered, L, root_idx, dropout)
        self.d = d
        # Project scalar gene expression into d dims
        self.in_proj = nn.Linear(1, d)
        # Shared per-layer transform
        self.W = nn.Parameter(torch.empty(L, d, d))
        nn.init.xavier_uniform_(self.W)
        self.bias = nn.Parameter(torch.zeros(N_layered, d))
        self.head = nn.Linear(root_idx.numel() * d, 2)

    def forward(self, X_gene_batch):
        B = X_gene_batch.size(0)
        h = torch.zeros((B, self.N, self.d), device=X_gene_batch.device, dtype=X_gene_batch.dtype)

        gene_vals = X_gene_batch.unsqueeze(-1)  # [B, G, 1]
        gene_emb = self.in_proj(gene_vals)      # [B, G, d]
        h[:, gene_layered_ids.to(h.device), :] = gene_emb

        for l in range(self.L):
            st = layer_structs[l]
            src = st["src"].to(h.device)
            dst_unique = st["dst_unique"].to(h.device)
            dst_pos = st["dst_pos"].to(h.device)

            h_src = h.index_select(1, src)  # [B, E, d]
            # Linear transform per layer
            Wl = self.W[l]                  # [d,d]
            msg = torch.matmul(h_src, Wl)   # [B,E,d]

            # Aggregate into dst_unique
            agg = torch.zeros((B, dst_unique.numel(), self.d), device=h.device, dtype=h.dtype)
            # index_add over dim=1 expects same trailing dims
            agg.index_add_(1, dst_pos, msg)

            h_dst = torch.tanh(agg + self.bias.index_select(0, dst_unique).unsqueeze(0))
            h_dst = self.dropout(h_dst)
            h[:, dst_unique, :] = h_dst

        feat = self.readout_roots_flat(h)  # [B, 29*d]
        return self.head(feat)

class DenseMLPBaseline(nn.Module):
    def __init__(self, G, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(G, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    def forward(self, X_gene_batch):
        return self.net(X_gene_batch)

def build_model(name:str):
    if name == "binn_exact_d1":
        return BINNExactD1(N_layered=N_layered, E_layered=E_layered, L=L, root_idx=root_idx, dropout=DROPOUT)
    if name == "gcn_shared_d1":
        return GCNSharedD1(N_layered=N_layered, L=L, root_idx=root_idx, dropout=DROPOUT)
    if name == "gat_gate_d1":
        return GATGateD1(N_layered=N_layered, L=L, root_idx=root_idx, dropout=DROPOUT)
    if name == "vector_shared_d16":
        return VectorSharedD(N_layered=N_layered, L=L, root_idx=root_idx, d=D_VECTOR, dropout=DROPOUT)
    if name == "dense_mlp_baseline":
        return DenseMLPBaseline(G=G, dropout=DROPOUT)
    raise ValueError(f"Unknown model: {name}")

print("Models available:", MODELS_TO_RUN)

## Phase 7–9, Section 3: Training + evaluation utilities

We train each selected model and collect:
- per-epoch loss curves
- ROC-AUC, PR-AUC, balanced accuracy
- ROC and PR curves
- confusion matrix

Then we plot comparisons across models.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix, roc_curve, precision_recall_curve

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

def eval_model(model):
    model.eval()
    ys = []
    probs = []
    preds = []
    losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb, weight=class_weights)
            p = torch.softmax(logits, dim=-1)[:,1]
            pred = logits.argmax(dim=-1)
            ys.append(yb.detach().cpu())
            probs.append(p.detach().cpu())
            preds.append(pred.detach().cpu())
            losses.append(loss.detach().cpu())
    y_true = torch.cat(ys).numpy()
    p1 = torch.cat(probs).numpy()
    y_pred = torch.cat(preds).numpy()
    loss_mean = float(torch.stack(losses).mean())

    # metrics
    auc = roc_auc_score(y_true, p1)
    auprc_pos = average_precision_score(y_true, p1)  # tumor=1
    auprc_neg = average_precision_score(1-y_true, 1-p1)  # normal=0
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    fpr, tpr, _ = roc_curve(y_true, p1)
    prec, rec, _ = precision_recall_curve(y_true, p1)

    return {
        "val_loss": loss_mean,
        "auc": float(auc),
        "auprc_tumor": float(auprc_pos),
        "auprc_normal": float(auprc_neg),
        "bal_acc": float(bal_acc),
        "cm": cm,
        "roc": (fpr, tpr),
        "pr": (prec, rec),
    }

def train_one(model_name):
    model = build_model(model_name).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    hist = {"train_loss": [], "val_loss": [], "auc": [], "bal_acc": [], "auprc_tumor": [], "auprc_normal": []}
    best = None
    best_state = None

    for epoch in range(1, EPOCHS+1):
        model.train()
        total = 0.0
        n = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb, weight=class_weights)
            loss.backward()
            opt.step()
            total += float(loss.detach().cpu()) * xb.size(0)
            n += xb.size(0)

        train_loss = total / max(n,1)
        metrics = eval_model(model)

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(metrics["val_loss"])
        hist["auc"].append(metrics["auc"])
        hist["bal_acc"].append(metrics["bal_acc"])
        hist["auprc_tumor"].append(metrics["auprc_tumor"])
        hist["auprc_normal"].append(metrics["auprc_normal"])

        print(f"[{model_name}] epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {metrics['val_loss']:.4f} | auc {metrics['auc']:.4f} | bal_acc {metrics['bal_acc']:.4f}")

        # Track best by AUC
        if best is None or metrics["auc"] > best["auc"]:
            best = metrics
            best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}

    # Save checkpoint
    run_dir = OUT_DIR / "runs_compare"
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / f"{model_name}_seed{SEED}_epochs{EPOCHS}.pt"
    torch.save({
        "model_name": model_name,
        "state_dict": best_state,
        "config": {
            "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE, "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY, "DROPOUT": DROPOUT,
            "L": L, "R": R, "G": G, "N_layered": N_layered, "E_layered": E_layered, "D_VECTOR": D_VECTOR,
        },
        "best_metrics": best,
        "history": hist,
    }, ckpt_path)
    print("Saved:", ckpt_path)

    return best, hist, ckpt_path

## Phase 7–9, Section 4: Run training for selected models

This will train each model listed in `MODELS_TO_RUN` and collect results.

What you should see:
- a line per epoch per model with train loss / val loss / AUC / balanced accuracy
- a saved checkpoint file per model

In [ ]:
results = {}
histories = {}
ckpts = {}

for name in MODELS_TO_RUN:
    print("\n" + "="*80)
    best, hist, ckpt = train_one(name)
    results[name] = best
    histories[name] = hist
    ckpts[name] = str(ckpt)

print("\nDone. Models trained:", list(results.keys()))

## Phase 7–9, Section 5: Visual comparisons

We plot:
1) Loss curves (train/val)
2) Metric bar charts (AUC, balanced accuracy, PR-AUC for tumor/normal)
3) ROC curves overlay
4) PR curves overlay (tumor class)
5) Confusion matrices

These visuals make Phase 7–9 comparisons much easier to interpret.

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(histories):
    plt.figure()
    for name,h in histories.items():
        plt.plot(h["train_loss"], label=f"{name} train")
        plt.plot(h["val_loss"], linestyle="--", label=f"{name} val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Loss curves")
    plt.legend()
    plt.show()

def plot_metric_bars(results, key, title):
    names = list(results.keys())
    vals = [results[n][key] for n in names]
    plt.figure()
    plt.bar(names, vals)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(key)
    plt.title(title)
    plt.show()

def plot_roc(results):
    plt.figure()
    for name,res in results.items():
        fpr, tpr = res["roc"]
        plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("ROC curves (val)")
    plt.legend()
    plt.show()

def plot_pr(results):
    plt.figure()
    for name,res in results.items():
        prec, rec = res["pr"]
        plt.plot(rec, prec, label=f"{name} (AUPRC_tumor={res['auprc_tumor']:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall curves (tumor=positive, val)")
    plt.legend()
    plt.show()

def plot_confusion_matrices(results):
    names = list(results.keys())
    for name in names:
        cm = results[name]["cm"]
        plt.figure()
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Confusion matrix (val): {name}")
        plt.colorbar()
        plt.xticks([0,1], ["normal(0)","tumor(1)"])
        plt.yticks([0,1], ["normal(0)","tumor(1)"])
        for i in range(2):
            for j in range(2):
                plt.text(j, i, str(cm[i,j]), ha="center", va="center")
        plt.ylabel("True")
        plt.xlabel("Pred")
        plt.show()

plot_loss_curves(histories)
plot_metric_bars(results, "auc", "Validation ROC-AUC")
plot_metric_bars(results, "bal_acc", "Validation Balanced Accuracy")
plot_metric_bars(results, "auprc_tumor", "Validation PR-AUC (tumor=positive)")
plot_metric_bars(results, "auprc_normal", "Validation PR-AUC (normal=positive)")
plot_roc(results)
plot_pr(results)
plot_confusion_matrices(results)

print("Checkpoints:", json.dumps(ckpts, indent=2))

## Phase 9 (optional): Official BINN package baseline

This is optional and may require extra dependency setup.

If you want, you can install the official BINN package from GitHub and attempt to train its baseline.
Because our Phase 6 proved the masked-FFNN view is equivalent to the layered MPNN view, the main value of this section is *benchmarking against their training defaults and interpretability pipeline*.

If you skip this, you still have a faithful BINN-equivalent model: `binn_exact_d1`.

In [ ]:
# Optional: install BINN package (commented out by default)
# !pip -q install git+https://github.com/infectionmedicineproteomics/binn

print("Phase 9 optional section: not executed by default.")
print("If you want to use the official binn package, uncomment the pip install and we can adapt inputs to its expected format.")